# Part 0: Train SpiderNet on the PerturbFISH dataset

This notebook loads the **processed PerturbFISH SpiderNet inputs** generated by `spidernet_dataloading_MIdimselection_PerturbFISH`, trains **SpiderNet**, and exports the key outputs required for downstream analysis.  
Its main goal is to learn **latent meta-interactions (MIs)** from the data and to summarize the **intercellular signaling programs** captured by the trained model.

Specifically, this notebook:

1. **Sets up the analysis configuration**  
   Define the **dataset paths**, the **processed-data directory**, ligand–receptor resources, and **model hyperparameters** used throughout the workflow.

2. **Loads the processed PerturbFISH data**  
   Reuse the outputs already generated by `spidernet_dataloading_MIdimselection_PerturbFISH` instead of rerunning the legacy preprocessing module inside this notebook.

3. **Trains SpiderNet on the PerturbFISH dataset**  
   Fit the model to learn structured representations of **cell–cell communication** and the underlying **meta-interaction programs**.

4. **Infers latent MI representations and loadings**  
   Export the learned **MI factors**, along with associated **gene** and **ligand–receptor (LR) loadings**, for downstream interpretation and comparison.

5. **Performs enrichment analysis for biological interpretation**  
   Run enrichment analysis on the learned LR-associated programs to help interpret the **biological signaling patterns** captured by SpiderNet.

Overall, this notebook serves as the **training and representation-learning stage** of the PerturbFISH analysis pipeline. It produces the trained model and the core intermediate outputs used in later **baseline comparison**, **functional interpretation**, and **in silico perturbation** analyses.


## 1. Imports and device setup


In [ ]:
import json
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from IPython.display import display

from SpiderNet.utils import *
from SpiderNet.config import *
from SpiderNet.io import load_processed_data
from SpiderNet.api import build_model, run_training

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


## 2. Configure input and output paths


In [ ]:
from run_benchmarks import DATA_ROOT, UPSTREAM_ROOT, PROCESSED_ROOT, OUTPUT_DIR
OUTPUT_ROOT = UPSTREAM_ROOT
PROCESSED_DATA_DIR = PROCESSED_ROOT

# Species used for the ligand–receptor databases.
SPECIES = "human"  # "mouse" or "human"

# These preprocessing parameters are recorded for reference.
# The processed bundle is expected to have already been generated by
# `spidernet_dataloading_MIdimselection_PerturbFISH`.
N_HVG = 1000
N_HVG_LR = 2000
NUM_NEIGHBORS = 10

# Training parameters
DIM_ENVIR = 23  # number of latent MI dimensions
N_JOBS = 10  # number of parallel CPU cores for initialization and training
MAX_EPOCH = 20000  # maximum number of training epochs

VERSION = "V1"

paths = PathConfig(
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    version=VERSION,
    species=SPECIES,
)

SAMPLE_ID_COL = "sample_name"
CELL_TYPE_COL = "celltype2"
SPATIAL_KEY = "spatial"

config = {
    "DATA_ROOT": str(DATA_ROOT),
    "OUTPUT_ROOT": str(OUTPUT_ROOT),
    "PROCESSED_DATA_DIR": str(PROCESSED_DATA_DIR),
    "SPECIES": SPECIES,
    "SAMPLE_ID_COL": SAMPLE_ID_COL,
    "CELL_TYPE_COL": CELL_TYPE_COL,
    "SPATIAL_KEY": SPATIAL_KEY,
    "N_HVG": N_HVG,
    "N_HVG_LR": N_HVG_LR,
    "NUM_NEIGHBORS": NUM_NEIGHBORS,
    "DIM_ENVIR": DIM_ENVIR,
    "N_JOBS": N_JOBS,
    "MAX_EPOCH": MAX_EPOCH,
    "VERSION": VERSION,
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

config_path = OUTPUT_DIR / "config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

print(f"Config saved to: {config_path}")


## 3. Configure training parameters and the processed-data source


In [ ]:
preprocess_cfg = PreprocessConfig(
    n_hvg=N_HVG,
    n_hvg_lr=N_HVG_LR,
    num_neighbors=NUM_NEIGHBORS,
)

train_cfg = TrainingConfig(
    version=VERSION,
    dim_envir=DIM_ENVIR,
    max_epoch=MAX_EPOCH,
    n_jobs=N_JOBS,
)

processed_data_dir = Path(PROCESSED_DATA_DIR)

base_run_dir = OUTPUT_DIR
base_model_dir = OUTPUT_DIR / "Model"
base_run_dirs = {"run_dir": base_run_dir, "model_dir": base_model_dir}

for p in [base_run_dir, base_model_dir]:
    p.mkdir(parents=True, exist_ok=True)

print(base_run_dirs)

with open(base_run_dir / "run_dirs.json", "w", encoding="utf-8") as handle:
    json.dump({k: str(v) for k, v in base_run_dirs.items()}, handle, indent=2)

processed_data_info = {
    "processed_data_dir": str(processed_data_dir),
    "generated_by": "spidernet_dataloading_MIdimselection_PerturbFISH",
}
with open(base_run_dir / "processed_data_source.json", "w", encoding="utf-8") as handle:
    json.dump(processed_data_info, handle, indent=2)


## 4. Load the processed PerturbFISH SpiderNet bundle

This notebook assumes that `spidernet_dataloading_MIdimselection_PerturbFISH` has already been run and that the processed files are stored under `PROCESSED_DATA_DIR`.


In [ ]:
if not processed_data_dir.exists():
    raise FileNotFoundError(
        f"Processed data directory does not exist: {processed_data_dir}\n"
        "Please run `spidernet_dataloading_MIdimselection_PerturbFISH` first or update PROCESSED_DATA_DIR."
    )

required_processed_files = [
    "adata_all.h5ad",
    "adata_list.pkl",
    "SpiderNet_data_pyg_list.pkl",
    "LR_list.pkl",
    "LR_list_cellchatdb.pkl",
    "LR_meta_cellchatdb.pkl",
    "genenames_train.pkl",
]
missing_processed_files = [
    file_name for file_name in required_processed_files
    if not (processed_data_dir / file_name).exists()
]

if missing_processed_files:
    raise FileNotFoundError(
        "The processed data bundle is incomplete under "
        f"{processed_data_dir}. Missing files: {missing_processed_files}"
    )

print(f"Using processed data from: {processed_data_dir}")


## 5. Load the processed SpiderNet dataset


In [ ]:
def sanity_check_processed(processed):
    check_df = pd.DataFrame(
        {
            "n_cells": [processed.adata_list[i].obs.shape[0] for i in range(len(processed.adata_list))],
            "edge_index_max": [
                torch.max(processed.spidernet_data[i]["edge_index"]).cpu().item()
                for i in range(len(processed.adata_list))
            ],
        }
    )
    mismatch = check_df["n_cells"] != (check_df["edge_index_max"] + 1)
    return check_df, mismatch


processed = load_processed_data(processed_data_dir)

print("Processed data dir:", processed_data_dir)
print("Number of batches:", len(processed.spidernet_data))
print("Number of LR pairs:", len(processed.lr_list))
print("Number of training genes:", processed.genenames_train.shape[0])

check_df, mismatch = sanity_check_processed(processed)
if mismatch.any():
    print("Potential data-loading mismatch detected:")
    display(check_df.loc[mismatch])
else:
    print("Sanity check passed.")

check_df


## 6. Build and train the SpiderNet model


In [ ]:
time_start_total = time.time()

model = build_model(
    processed=processed,
    train_cfg=train_cfg,
    device=device,
)

model = run_training(
    model=model,
    processed=processed,
    train_cfg=train_cfg,
    model_dir=base_model_dir,
    device=device,
)

# Save configs for reproducibility.
with open(base_model_dir / "SpiderNet_model_config.json", "w", encoding="utf-8") as handle:
    json.dump(train_cfg.to_dict(), handle, indent=2)

with open(base_model_dir / "SpiderNet_preprocess_config.json", "w", encoding="utf-8") as handle:
    json.dump(preprocess_cfg.to_dict(), handle, indent=2)

time_end_total = time.time()
print(f"Total modeling time cost: {(time_end_total - time_start_total) / 60:.2f} minutes")


## 7. Save the training summary


In [ ]:
final_model_path = base_model_dir / f"model_epoch{train_cfg.max_epoch - 1}.pth"

training_summary_df = pd.DataFrame(
    [
        {
            "processed_data_dir": str(processed_data_dir),
            "run_dir": str(base_run_dir),
            "model_dir": str(base_model_dir),
            "final_model_exists": final_model_path.exists(),
            "n_batches": len(processed.spidernet_data),
            "n_lr": len(processed.lr_list),
            "n_genes": int(processed.genenames_train.shape[0]),
            "dim_envir": train_cfg.dim_envir,
        }
    ]
)

training_summary_path = base_run_dir / "training_summary_part0.csv"
training_summary_df.to_csv(training_summary_path, index=False)
print(f"Saved summary to: {training_summary_path}")

training_summary_df


## 8. Define helper functions for meta-interaction inference and export


In [ ]:
def infer_meta_interactions(model, processed, device):
    """
    Run the trained SpiderNet model on all batches and collect:
    - batch-wise meta-interaction scores
    - global LR loading
    - global sender loading
    - global receiver loading

    The normalization follows the original Part0 notebook:
    each loading matrix is scaled by the maximum value of each MI across cells,
    and each batch-wise MI matrix is normalized by the same MI-wise maxima.
    """
    model = model.to(device)
    model.eval()

    factor_envir_list = []

    with torch.no_grad():
        for batch_data in processed.spidernet_data:
            outputs = model(batch_data.to(device))
            _, _, _, _, factor_envir_curbatch, loading_receiver, loading_sender, loading_LR = outputs
            factor_envir_list.append(factor_envir_curbatch.detach().cpu().numpy().astype(np.float32))

    factor_envir_use = np.vstack(factor_envir_list).astype(np.float32)
    loading_LR_use = loading_LR.detach().cpu().numpy().astype(np.float32)
    loading_receiver_use = loading_receiver.detach().cpu().numpy().astype(np.float32)
    loading_sender_use = loading_sender.detach().cpu().numpy().astype(np.float32)

    factor_envir_use_max = np.max(factor_envir_use, axis=0)

    loading_LR_use = loading_LR_use * factor_envir_use_max[:, np.newaxis]
    loading_receiver_use = loading_receiver_use * factor_envir_use_max[:, np.newaxis]
    loading_sender_use = loading_sender_use * factor_envir_use_max[:, np.newaxis]

    factor_envir_use = factor_envir_use / (factor_envir_use_max[np.newaxis, :] + 1e-10)
    factor_envir_list = [
        factor_cur / (factor_envir_use_max[np.newaxis, :] + 1e-10)
        for factor_cur in factor_envir_list
    ]

    return {
        "Factor_envir_use": factor_envir_use,
        "Factor_envir_list": factor_envir_list,
        "loading_LR_use": loading_LR_use,
        "loading_receiver_use": loading_receiver_use,
        "loading_sender_use": loading_sender_use,
    }


def save_meta_interaction_outputs(mi_outputs, processed, output_dir):
    """
    Save the inferred meta-interaction matrices and loadings.
    The filenames are kept consistent with the original Part0 notebook.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    gene_names = np.array(processed.genenames_train)
    mi_names = [f"MI{i + 1}" for i in range(mi_outputs["loading_receiver_use"].shape[0])]

    loading_receiver_use_df = pd.DataFrame(
        mi_outputs["loading_receiver_use"],
        index=mi_names,
        columns=gene_names,
    )
    loading_sender_use_df = pd.DataFrame(
        mi_outputs["loading_sender_use"],
        index=mi_names,
        columns=gene_names,
    )

    loading_receiver_use_df_path = output_dir / "loading_receiver_use.csv"
    loading_sender_use_df_path = output_dir / "loading_sender_use.csv"
    Factor_envir_use_path = output_dir / "Factor_envir_use.npy"
    Factor_envir_list_path = output_dir / "Factor_envir_list.pkl"
    loading_LR_use_path = output_dir / "loading_LR_use.npy"

    loading_receiver_use_df.to_csv(loading_receiver_use_df_path, index=True)
    loading_sender_use_df.to_csv(loading_sender_use_df_path, index=True)
    np.save(Factor_envir_use_path, mi_outputs["Factor_envir_use"])
    np.save(loading_LR_use_path, mi_outputs["loading_LR_use"])

    with open(Factor_envir_list_path, "wb") as handle:
        pickle.dump(mi_outputs["Factor_envir_list"], handle)

    return {
        "loading_receiver_use_df_path": loading_receiver_use_df_path,
        "loading_sender_use_df_path": loading_sender_use_df_path,
        "Factor_envir_use_path": Factor_envir_use_path,
        "Factor_envir_list_path": Factor_envir_list_path,
        "loading_LR_use_path": loading_LR_use_path,
    }


## 9. Infer meta-interactions and loadings


In [ ]:
mi_outputs = infer_meta_interactions(
    model=model,
    processed=processed,
    device=device,
)

print("Factor_envir_use shape:", mi_outputs["Factor_envir_use"].shape)
print("loading_LR_use shape:", mi_outputs["loading_LR_use"].shape)
print("loading_receiver_use shape:", mi_outputs["loading_receiver_use"].shape)
print("loading_sender_use shape:", mi_outputs["loading_sender_use"].shape)


## 10. Save inferred meta-interactions and loadings


In [ ]:
mi_output_paths = save_meta_interaction_outputs(
    mi_outputs=mi_outputs,
    processed=processed,
    output_dir=base_run_dir,
)

with open(base_run_dir / "meta_output_paths.json", "w", encoding="utf-8") as handle:
    json.dump({k: str(v) for k, v in mi_output_paths.items()}, handle, indent=2)

mi_output_paths


## 11. Run LR-loading enrichment analysis


In [ ]:
from SpiderNet.analysis import LRLoading_enrichment

LRLoading_enrichment(
    loading_LR_use_path=base_run_dir / "loading_LR_use.npy",
    lr_list_path=processed_data_dir / "LR_list.pkl",
    lr_list_cellchatdb_path=processed_data_dir / "LR_list_cellchatdb.pkl",
    lr_meta_cellchatdb_path=processed_data_dir / "LR_meta_cellchatdb.pkl",
    Factor_envir_use_path=base_run_dir / "Factor_envir_use.npy",
    file_savepath_main=base_run_dir,
    show=True,
)
